# Benchmarking Models Across Datasets

Use `benchmark()` to compare feature models or comprehension-model
specifications across datasets and response measures. An `EvaluationSpec` assigns features to targets, predictors and controls; `AnalysisConfig`
supplies the response model and evaluation design. This example compares two
language models using the same features and analysis. Custom models follow the
same interface; see [Extending REPLICA](extending_replica.ipynb).

`benchmark()` applies each specification factory to each dataset and collects the results.

## Prerequisites

```bash
pip install -e ".[models,r]"
replica fetch provo natural_stories
```

The `models` extra installs torch and transformers for the GPT-2 models.
The default mixed model requires system R with `lme4`; see [installation](getting_started.ipynb).


In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s")

## 1. Load datasets

In [ ]:
from replica.dataset import load_dataset

provo = load_dataset("provo")
ns = load_dataset("natural_stories")

for ds in [provo, ns]:
    print(
        f"{ds.info.name:20s}  stimuli={len(ds.stimuli()):4d}  "
        f"participants={len(ds.participants):3d}  measures={ds.measures}"
    )

### Runtime

These examples evaluate the complete loaded datasets and run language models. Runtime depends on the model and hardware. Start with one dataset and one model when checking the workflow. Use documented loader options where a source supports selecting a cohort; editing internal stimulus and response lists can leave source references and fixation caches inconsistent.

## 2. Bind specifications to each dataset

A specification factory receives the loaded dataset and selects its target.
Each model-backed feature names its own model source. `benchmark()` calls each
factory for each dataset; it never substitutes the source of an already bound feature.

`Surprisal()` and `Entropy()` use bits by default. Here `Entropy()` pools the
next-subword distribution by surface form; it does not estimate whole-word
entropy. The analysis uses raw responses, training-fold feature scaling and
the default mixed-effects model, with up to five folds on each holdout axis.


In [ ]:
from functools import partial

from replica import AnalysisConfig, EvaluationSpec, ModelFeature, benchmark
from replica.feature import Entropy, HuggingFaceLM, Surprisal, UnitLength


def spec_for(dataset, *, model):
    return EvaluationSpec(
        targets=[dataset.feature(dataset.default_measure)],
        predictors=[ModelFeature(model, Surprisal()), ModelFeature(model, Entropy())],
        controls=[UnitLength()],
    )


results = benchmark(
    datasets=[provo, ns],
    evaluation_specs=[
        partial(spec_for, model=HuggingFaceLM("gpt2")),
        partial(spec_for, model=HuggingFaceLM("gpt2-medium")),
    ],
    analysis=AnalysisConfig(n_folds=5),
)

## 3. Summarize results

The results are a flat list of `AnalysisResult` objects. Each has `.dataset_name`, `.model_name`, `.feature_name`, and `.dv_name` so you can slice and group as needed. Both specs add surprisal and entropy jointly, so `feature_name` names the joint comparison and `AnalysisResult.coefficients` holds one slope per predictor.

`AnalysisResult.response` records the resolved response source and calculation where available. Include it when exporting results; a DV name alone does not distinguish alternative algorithms. Log-likelihood gains are in nats, independently of the predictor units.

In [ ]:
# Print a table of results
print(
    f"{'Dataset':20s} {'Model':15s} {'Feature':12s} {'DV':18s} {'delta-LL':>10s} {'dLL/unit':>12s}"
)
print("-" * 90)
for r in results:
    print(
        f"{r.dataset_name:20s} {r.model_name:15s} {r.feature_name:12s} "
        f"{r.dv_name:18s} {r.delta_ll:+10.2f} {r.delta_ll_per_unit:+12.6f}"
    )

## 4. Per-dataset targets

Targets are explicit and nonempty. The factory above selects each dataset's
declared default measurement. To select other quantities, build specifications
bound to the corresponding dataset and call `evaluate()` directly:


In [ ]:
from replica import evaluate

# PROVO: evaluate two eye-tracking DVs
provo_spec = EvaluationSpec(
    predictors=[ModelFeature(HuggingFaceLM("gpt2"), Surprisal())],
    controls=[UnitLength()],
    targets=[provo.feature("first_run_in_first_pass"), provo.feature("first_fixation")],
)

# Natural Stories: evaluate SPR reading time
ns_spec = EvaluationSpec(
    predictors=[ModelFeature(HuggingFaceLM("gpt2"), Surprisal())],
    controls=[UnitLength()],
    targets=[ns.feature("spr_rt")],
)

config = AnalysisConfig(n_folds=5)  # Up to five folds per axis.

results_provo = evaluate(provo, provo_spec, config)
results_ns = evaluate(ns, ns_spec, config)

for r in results_provo + results_ns:
    print(f"{r.dataset_name:20s} {r.dv_name:18s} delta-LL={r.delta_ll:+.2f}")

## 5. Working with results as a DataFrame

For larger benchmarks, converting to a pandas DataFrame makes analysis easier.

In [ ]:
import pandas as pd

df = pd.DataFrame(
    [
        {
            "dataset": r.dataset_name,
            "model": r.model_name,
            "feature": r.feature_name,
            "dv": r.dv_name,
            "response": r.response,
            "delta_ll": r.delta_ll,
            "delta_ll_per_unit": r.delta_ll_per_unit,
            "delta_r2": r.delta_r2,
            "n": r.n_observations,
        }
        for r in results
    ]
)

df